# Task 2.2: Reproduction of Explicit Data Mapping + Linear SVM

*   **Contribution Reproduced:** The explicit transformation of input features into a degree-2 polynomial space followed by classification using a fast Large-Scale Linear SVM (as opposed to dual decomposition kernel trick) as shown in Section 3.
*   **Evaluation Metric:** Testing Accuracy (percentage of correct classifications).

In [1]:
import numpy as np
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

# Load data from 2.1 conceptually (we regenerate it here for self-contained execution)
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
X, y = make_circles(n_samples=500, noise=0.1, factor=0.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

The code below explicitly maps the 2D data into a higher dimensional polynomial space. This directly corresponds to **Equation (5)** in Section 3.1 of the paper, where the authors construct the expanded $\phi(x)$ vector.

In [2]:
# Step 1: Explicit mapping (Degree-2 Polynomial)
# This generates [1, x1, x2, x1^2, x1*x2, x2^2] replacing the kernel evaluation.
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_mapped = poly.fit_transform(X_train)
X_test_mapped = poly.transform(X_test)

print(f'Original features: {X_train.shape[1]}, Mapped features: {X_train_mapped.shape[1]}')

Original features: 2, Mapped features: 5


The next block applies a pure Linear SVM (specifically `LibLinear` via `LinearSVC`) directly to the newly mapped $\phi(x)$ data. This references **Section 3.2** of the paper, where a linear solver is utilized to process the explicitly mapped data $O(\hat{n})$ instead of executing $O(l ar{n})$ complex dual kernel functions.

In [3]:
# Step 2: Training a Linear SVM on the explicitly mapped features 
# This uses LIBLINEAR internally under sklearn.
start_time = time.time()
linear_svm_explicit = LinearSVC(C=1.0, random_state=42, dual=False)
linear_svm_explicit.fit(X_train_mapped, y_train)
train_time_explicit = time.time() - start_time

# Prediction metric
y_pred_explicit = linear_svm_explicit.predict(X_test_mapped)
acc_explicit = accuracy_score(y_test, y_pred_explicit)

print(f'Explicit Poly + LinearSVC Accuracy: {acc_explicit * 100:.2f}%')
print(f'Training time: {train_time_explicit:.5f} sec')

Explicit Poly + LinearSVC Accuracy: 99.00%
Training time: 0.00181 sec
